# PARTIE A — Chargement & choix des variables 

### ÉTAPE 1   Charger le fichier Excel et le convertir en CSV 
- Chargez data/dataset_assurance_ML.xlsx avec pd.read_excel (bibliothèque openpyxl) 
- Sauvegardez-le en CSV avec l'encodage utf-8-sig pour la suite du projet 
- Rechargez le CSV et vérifiez la shape

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

  
df = pd.read_excel('../data/dataset_assurance_ML.xlsx') 
df.to_csv('../data/dataset_assurance_ML.csv', index=False, encoding='utf-8-sig') 
  
df = pd.read_csv('../data/dataset_assurance_ML.csv', encoding='utf-8-sig') 
print(df.shape) 

(500, 27)


In [2]:
df.isnull().sum() # 0
df.duplicated().sum() # 0

0

In [3]:
df.head()

,N° Police,Nom,Prénom,Sexe,Âge,Catégorie Prof.,Salaire Annuel (€),Ville,Code Postal,Type Contrat,...,Type Véhicule,Usage Véhicule,Puissance Fiscale (CV),Valeur Véhicule (€),Coeff. Bonus-Malus,Nb Sinistres (3 ans),Montant Sinistres (€),Dernier Sinistre,Score Risque (0-100),Résiliation
0,ASS01000,Bernard,Valérie,F,56,Profession libérale,76288,Nice,6200,Gold,...,SUV,Loisirs,9,20108,0.80,1,1447,Dégât des eaux,12,0
1,ASS01001,Simon,Éric,H,69,Employé,30020,Paris,75018,Bronze,...,SUV,Loisirs,9,29845,1.02,1,685,Incendie,20,0
2,ASS01002,Lambert,Inès,F,46,Technicien,36995,Lyon,69009,Silver,...,Citadine,Domicile-Travail,4,19035,0.54,0,0,Aucun,0,0
3,ASS01003,Blanc,François,H,32,Profession libérale,64773,Bordeaux,33200,Gold,...,Berline,Loisirs,8,25765,0.90,1,0,Vol,28,1
4,ASS01004,Vincent,Patrick,H,60,Cadre,69870,Nantes,44200,Gold,...,Berline,Loisirs,7,24365,0.96,1,0,Incendie,7,0


### ÉTAPE 2   La variable cible 
- Affichez la répartition de 'Résiliation' en effectifs puis en pourcentage 
- Question : le problème est-il équilibré ? Quelle conséquence pour l'entraînement ? 

In [4]:
TARGET = 'Résiliation' 
print(df[TARGET].value_counts()) 
print(df[TARGET].value_counts(normalize=True).round(2))

Résiliation
0    450
1     50
Name: count, dtype: int64
Résiliation
0    0.9
1    0.1
Name: proportion, dtype: float64


Non le probleme n'est pas equilibre

### ÉTAPE 3   Détecter une fuite de données (data leakage) 
- Croisez 'Statut Contrat' et 'Résiliation' avec pd.crosstab 
- Question : que remarquez-vous ? Peut-on utiliser 'Statut Contrat' pour prédire la résiliation ?

In [5]:
print(pd.crosstab(df['Statut Contrat'], df[TARGET])) 

Résiliation       0   1
Statut Contrat         
Actif           450   0
Résilié           0  36
Suspendu          0  14


On remarque le statut vient aprs un etat resiliation de resiliation ou non

### ÉTAPE 4   Choisir les variables numériques et catégorielles 
- Définissez num_cols (8 variables) et cat_cols (4 variables) selon les listes ci-dessous 
- Les variables sont choisies d'après l'EDA du Jour 2 (les plus liées à la cible) et pour garder une 
interface de saisie raisonnable : 12 champs 
- Construisez X et y 

In [6]:
num_cols = ['Âge', 'Salaire Annuel (€)', 'Prime Annuelle (€)', 'Ancienneté (mois)', 
            'Coeff. Bonus-Malus', 'Nb Sinistres (3 ans)', 
            'Montant Sinistres (€)', 'Score Risque (0-100)'] 
cat_cols = ['Type Contrat', 'Catégorie Prof.', 'Usage Véhicule', 'Dernier Sinistre'] 
X = df[num_cols + cat_cols] 
y = df[TARGET] 
print(X.shape, y.shape) 

(500, 12) (500,)


### ÉTAPE 5   Vérifier le lien avec la cible 
- Calculez la corrélation de chaque variable numérique avec la cible (corrwith) et triez 
- Calculez le taux de résiliation par modalité de 'Dernier Sinistre' 
- Question : quelles sont les 3 variables numériques les plus liées à la résiliation ? 

In [7]:
print(X[num_cols].corrwith(y).round(3).sort_values(ascending=False)) 
print(df.groupby('Dernier Sinistre')[TARGET].mean().round(2).sort_values()) 

Nb Sinistres (3 ans)     0.455
Score Risque (0-100)     0.441
Coeff. Bonus-Malus       0.412
Montant Sinistres (€)    0.284
Prime Annuelle (€)       0.041
Âge                     -0.006
Salaire Annuel (€)      -0.012
Ancienneté (mois)       -0.055
dtype: float64
Dernier Sinistre
Aucun                    0.03
Incendie                 0.16
Catastrophe naturelle    0.18
Accident                 0.26
Bris de glace            0.30
Vol                      0.30
Dégât des eaux           0.36
Name: Résiliation, dtype: float64


Bris de glace, Vol, Degat des eaux

# PARTIE B — Pipeline, entraînement & évaluation


#### � Pourquoi un Pipeline ? 
Au Jour 2, vous avez appliqué scaler et encodeur « à la main ». Problème : pour prédire sur un nouveau 
client, il faut refaire exactement les mêmes opérations, dans le même ordre, avec les mêmes objets 
fittés. Le Pipeline scikit-learn encapsule tout cela : on lui donne des données brutes, il s'occupe du 
reste. C'est ce qui rendra l'interface de la partie D triviale à écrire. 

### ÉTAPE 6   Séparer train / test 
- Faites un train_test_split à 80/20, random_state=42, en conservant le ratio de la cible (stratify) 
- Vérifiez que le taux de résiliation est bien ~10 % dans les deux jeux 

In [8]:
from sklearn.model_selection import train_test_split 
  
X_train, X_test, y_train, y_test = train_test_split( 
    X, y, test_size=0.2, random_state=42, stratify=y) 
print(X_train.shape, X_test.shape) 
print(y_train.mean().round(2), y_test.mean().round(2)) 

(400, 12) (100, 12)
0.1 0.1


### ÉTAPE 7   Le prétraitement en un seul objet : ColumnTransformer 
- Créez un ColumnTransformer qui applique StandardScaler aux num_cols et OneHotEncoder aux 
cat_cols 
- handle_unknown='ignore' : une modalité jamais vue à l'entraînement ne fera pas planter l'application 

In [9]:
from sklearn.compose import ColumnTransformer 
from sklearn.preprocessing import StandardScaler, OneHotEncoder 
  
preprocessor = ColumnTransformer([ 
    ('num', StandardScaler(), num_cols), 
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols), 
])

### ÉTAPE 8   Deux candidats dans un Pipeline 
- Créez deux pipelines [prep → model] : une Régression Logistique et un Random Forest 
- Les deux avec class_weight='balanced' à cause du déséquilibre 90/10 
- Pour le Random Forest, limitez la profondeur (max_depth=4, min_samples_leaf=10) pour éviter le 
sur-apprentissage sur 400 lignes 

In [10]:
from sklearn.pipeline import Pipeline 
from sklearn.linear_model import LogisticRegression 
from sklearn.ensemble import RandomForestClassifier 
  
candidats = { 
    'Régression Logistique': LogisticRegression( 
        max_iter=1000, class_weight='balanced', random_state=42), 
    'Random Forest': RandomForestClassifier( 
        n_estimators=300, max_depth=4, min_samples_leaf=10, 
        class_weight='balanced', random_state=42), 
} 
pipelines = {nom: Pipeline([('prep', preprocessor), ('model', algo)]) 
             for nom, algo in candidats.items()} 

### ÉTAPE 9   Comparer par validation croisée 
  Pour chaque pipeline, calculez le ROC-AUC en validation croisée 5-fold sur le TRAIN uniquement 
- Affichez moyenne ± écart-type 
- Question : lequel retenez-vous ? L'écart est-il significatif ? 

In [11]:
from sklearn.model_selection import cross_val_score 

for nom, pipe in pipelines.items(): 
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='roc_auc') 
    print(f'{nom:22s} AUC = {scores.mean():.3f} ± {scores.std():.3f}') 

Régression Logistique  AUC = 0.745 ± 0.113
Random Forest          AUC = 0.801 ± 0.096


### ÉTAPE 10   Entraîner le modèle retenu et évaluer sur le test 
- Entraînez le pipeline Random Forest sur X_train / y_train 
- Prédisez les classes ET les probabilités sur X_test 
- Calculez accuracy, F1, ROC-AUC et la matrice de confusion 

In [12]:
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, 
confusion_matrix, classification_report) 
pipeline = pipelines['Random Forest'] 
pipeline.fit(X_train, y_train) 
y_pred  = pipeline.predict(X_test) 
y_proba = pipeline.predict_proba(X_test)[:, 1] 
print('Accuracy :', accuracy_score(y_test, y_pred).round(3)) 
print('F1       :', f1_score(y_test, y_pred).round(3)) 
print('ROC-AUC  :', roc_auc_score(y_test, y_proba).round(3)) 
print(confusion_matrix(y_test, y_pred)) 
print(classification_report(y_test, y_pred, target_names=['Reste', 'Résilie'])) 

Accuracy : 0.87
F1       : 0.519
ROC-AUC  : 0.853
[[80 10]
 [ 3  7]]
              precision    recall  f1-score   support

       Reste       0.96      0.89      0.92        90
     Résilie       0.41      0.70      0.52        10

    accuracy                           0.87       100
   macro avg       0.69      0.79      0.72       100
weighted avg       0.91      0.87      0.88       100



### ÉTAPE 11   Lire la matrice de confusion comme un métier 
- Répondez dans une cellule Markdown : 
- Q1. Un modèle qui prédirait toujours « reste » aurait 90 % d'accuracy. Est-il meilleur que le vôtre ? 
- Q2. Pour le service Fidélisation, quelle erreur coûte le plus cher : rater un client qui va partir (faux 
négatif) ou appeler un client qui serait resté (faux positif) ? 
- Q3. Faut-il donc plutôt baisser ou monter le seuil de 0,5 ? 

### Reponses
1- Non, le model <<toujours reste>> une accuracy de 90%, le mien serait de 87%, avec un bon taux de recall 70%
2- Le faux négatif coûte généralement plus cher : c'est un client qui va réellement résilier mais que le modèle identifie comme un client qui va rester.

# PARTIE C — Sauvegarde & interrogation du modèle 

### ÉTAPE 12   Sauvegarder le pipeline complet 
- Sauvegardez le pipeline (prétraitement + modèle) dans models/pipeline_resiliation.pkl avec joblib 
- Vérifiez la taille du fichier créé 

In [13]:
import joblib, os 
joblib.dump(pipeline, '../models/pipeline_resiliation.pkl') 
print(os.path.getsize('../models/pipeline_resiliation.pkl') / 1024, 'Ko') 

449.4208984375 Ko


### ÉTAPE 13   Sauvegarder les métadonnées pour l'interface 
- L'interface aura besoin de connaître : le nom des colonnes, les bornes des variables numériques 
(pour les curseurs) et les modalités des catégorielles (pour les menus) 
- Construisez un dictionnaire meta et sauvegardez-le en JSON dans models/metadata.json 

In [14]:
import json 
meta = { 
    'modele': 'Random Forest', 
    'auc_test': round(float(roc_auc_score(y_test, y_proba)), 3), 
    'num_cols': num_cols, 
    'cat_cols': cat_cols, 
    'num_ranges': {c: {'min': float(X[c].min()), 'max': float(X[c].max()), 
                       'median': float(X[c].median())} for c in num_cols}, 
    'cat_values': {c: sorted(X[c].unique().tolist()) for c in cat_cols}, 
} 
with open('../models/metadata.json', 'w', encoding='utf-8') as f: 
    json.dump(meta, f, ensure_ascii=False, indent=2)

### ÉTAPE 14   Interroger le modèle sur un nouveau client 
- Rechargez le pipeline avec joblib.load (dans une NOUVELLE cellule, comme si vous repartiez de 
zéro) 
- Créez un DataFrame d'une ligne décrivant un client à risque (jeune, peu ancien, 3 sinistres, score 
élevé, contrat Bronze, dernier sinistre Vol) 
- Affichez la classe prédite et la probabilité de résiliation 
- Modifiez ensuite le profil (0 sinistre, ancienneté 200 mois, Gold, Aucun sinistre) et comparez 

In [15]:
modele = joblib.load('../models/pipeline_resiliation.pkl') 
  
client = pd.DataFrame([{ 
    'Âge': 34, 'Salaire Annuel (€)': 28000, 'Prime Annuelle (€)': 950, 
    'Ancienneté (mois)': 6, 'Coeff. Bonus-Malus': 1.25, 'Nb Sinistres (3 ans)': 3, 
    'Montant Sinistres (€)': 4200, 'Score Risque (0-100)': 72, 
    'Type Contrat': 'Bronze', 'Catégorie Prof.': 'Entrepreneur', 
    'Usage Véhicule': 'Professionnel', 'Dernier Sinistre': 'Vol', 
}]) 
print('Classe :', modele.predict(client)) 
print('Proba  :', modele.predict_proba(client)[0, 1].round(3)) 

Classe : [1]
Proba  : 0.816


# ÉTAPE 15   Provoquer l'erreur classique, puis passer en script 
- Supprimez la colonne 'Score Risque (0-100)' du DataFrame client et relancez la prédiction — lisez le 
message d'erreur 
- Question : quelle règle en tirez-vous pour l'interface ? 
- Enfin, regroupez tout le code des parties A, B, C (sans les affichages exploratoires) dans 
train_model.py et exécutez-le depuis le terminal 

In [16]:
try: 
    modele.predict(client.drop(columns=['Score Risque (0-100)'])) 
except Exception as e: 
    print('ERREUR :', e) 

# Terminal : 
# python train_model.py 

ERREUR : "['Score Risque (0-100)'] not in index"


# PARTIE D — Interface Streamlit 

### ÉTAPE 16   Première application : Hello Streamlit 
- Dans app.py, écrivez le code ci-dessous 
- Lancez-le depuis le terminal : streamlit run app.py 
- Le navigateur s'ouvre sur http://localhost:8501 

In [17]:
import streamlit as st 
  
st.title('🚗 Scoring de résiliation — Assurance Auto') 
st.write('Bonjour ! Mon premier modèle en ligne.') 

2026-09-19 13:38:09.273 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-19 13:38:09.918 
  command:

    streamlit run c:\Users\BC\anaconda3\envs\projet_data\lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-09-19 13:38:09.919 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-19 13:38:09.922 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-19 13:38:09.923 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-19 13:38:09.924 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-19 13:38:09.925 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


### ÉTAPE 17   Charger le modèle et les métadonnées (une seule fois) 
- Ajoutez la fonction charger_modele() ci-dessous, décorée avec @st.cache_resource 
- Configurez la page (titre d'onglet, mise en page large) avec st.set_page_config, qui doit être le 
PREMIER appel Streamlit 
- Affichez le nom du modèle et son AUC sous le titre avec st.caption 

In [18]:
import json, joblib 
import pandas as pd 
import streamlit as st 
  
st.set_page_config(page_title='Scoring Résiliation', page_icon='🚗', layout='wide') 
  
@st.____ 
def charger_modele(): 
    pipeline = joblib.load('models/pipeline_resiliation.pkl') 
    with open('models/metadata.json', encoding='utf-8') as f:
        meta = json.load(f) 
    return pipeline, meta 
  
pipeline, meta = charger_modele() 
num_cols, cat_cols = meta['num_cols'], meta['cat_cols'] 
rng, cats = meta['num_ranges'], meta['cat_values'] 
  
st.title('🚗 Scoring de résiliation — Assurance Auto') 
st.caption(f"Modèle : {meta['modele']}  ·  AUC test : {meta['auc_test']}") 

2026-09-19 13:38:10.142 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


AttributeError: module 'streamlit' has no attribute '____'

### ÉTAPE 18   Les curseurs des variables numériques 
- Écrivez la fonction curseur(col, step, fmt) qui crée un slider dans la barre latérale, borné par le 
min/max du metadata et positionné sur la médiane 
- Créez les 8 curseurs et stockez les valeurs dans un dictionnaire client 
- Le format '%d' affiche un entier ; renvoyez alors int(val)

In [ ]:
st.sidebar.header('👤 Profil du client') 
  
def curseur(col, step=1.0, fmt=None): 
    r = rng[col] 
    val = st.sidebar.slider(col, min_value=r['min'], max_value=r['max'], 
                            value=r['median'], step=step, format=fmt) 
    return int(val) if fmt == '%d' else val 
  
client = {} 
client['Âge']                   = curseur('Âge', 1.0, '%d') 
client['Salaire Annuel (€)']    = curseur('Salaire Annuel (€)', 500.0, '%d') 
client['Prime Annuelle (€)']    = curseur('Prime Annuelle (€)', 10.0, '%d') 
client['Ancienneté (mois)']     = curseur('Ancienneté (mois)', 1.0, '%d') 
client['Coeff. Bonus-Malus']    = curseur('Coeff. Bonus-Malus', 0.01, '%.2f') 
client['Nb Sinistres (3 ans)']  = curseur('Nb Sinistres (3 ans)', 1.0, '%d') 
client['Montant Sinistres (€)'] = curseur('Montant Sinistres (€)', 100.0, '%d') 
client['Score Risque (0-100)']  = curseur('Score Risque (0-100)', 1.0, '%d') 

### ÉTAPE 19   Les menus des variables catégorielles 
- Ajoutez un séparateur puis, en boucle sur cat_cols, un selectbox par variable alimenté par cats[col] 
- Question : pourquoi prendre les modalités depuis le metadata plutôt que de les écrire en dur ? 

In [ ]:
st.sidebar.markdown('---') 
for col in cat_cols: 
    client[col] = st.sidebar.selectbox(col, cats[col]) 

### ÉTAPE 20   Le bouton Prédire et le résultat 
- Ajoutez un bouton « Prédire » ; dans le bloc if, construisez un DataFrame d'une ligne à partir de 
client, dans l'ordre num_cols + cat_cols 
- Calculez la probabilité, affichez-la avec st.metric, une jauge st.progress, et un message coloré selon 
deux seuils 
- Ajoutez un st.info() dans le else pour guider l'utilisateur avant le premier clic 

In [ ]:
SEUIL_RISQUE = 0.55 
SEUIL_MODERE = 0.40 
  
if st.button('🔮 Prédire', type='primary', use_container_width=True): 
    df_client = pd.DataFrame([client])[num_cols + cat_cols]
    proba = float(pipeline.predict_proba(df_client)[0, 1])

    col1, col2 = st.columns([1, 2])
    with col1:
        st.metric('Probabilité de résiliation', f'{proba:.0%}')
        if proba >= SEUIL_RISQUE:
            st.error('⚠️ Client À RISQUE — action de rétention conseillée')
        elif proba >= SEUIL_MODERE:
            st.warning('🟠 Risque modéré — à surveiller') 
        else: 
            st.success('✅ Client fidèle — risque faible') 
    with col2: 
        st.write('Niveau de risque') 
        st.progress(proba) 
        st.write('Données envoyées au modèle :') 
        st.dataframe(df_client.T.astype(str).rename(columns={0: 'Valeur'}), use_container_width=True) 
else: 
    st.info('👈 Ajustez le profil dans la barre latérale, puis cliquez sur Prédire.') 

### ÉTAPE 21   Expliquer la prédiction : importance des variables 
- Toujours dans le bloc if, récupérez le Random Forest avec pipeline.named_steps['model'] et les 
noms de colonnes transformées avec pipeline.named_steps['prep'].get_feature_names_out() 
- Construisez une Series des 8 importances les plus fortes et affichez-la avec st.bar_chart 
- Les noms sont préfixés 'num__' ou 'cat__' : nettoyez-les avec split('__', 1)[1] 

In [ ]:
model = pipeline.named_steps['model'] 
if hasattr(model, 'feature_importances_'): 
    noms = pipeline.named_steps['prep'].get_feature_names_out() 
    imp = (pd.Series(model.feature_importances_, index=noms) 
           .sort_values(ascending=False).head(8)) 
    imp.index = [n.split('__', 1)[1] for n in imp.index] 
    st.subheader('📊 Les 8 variables les plus influentes du modèle') 
    st.bar_chart(imp) 